In [1]:
import h5py
import os
import numpy as np
import json
import shutil
import cv2
from PIL import Image

In [ ]:
annotation_folder = "/home/arisrei/research/RPL/ORION/annotations/human_demo/jones_golden_idol"
config_dict = {}
with open(os.path.join(annotation_folder, "config.json"), "r") as f:
    config_dict = json.load(f)

has_depth = False
with h5py.File(config_dict["original_file"], "r") as demo_file:
    rgb = demo_file["data/human_demo/obs/agentview_rgb/"][()]
    if "data/human_demo/obs/agentview_depth/" in demo_file:
        has_depth = True
        depth = demo_file["data/human_demo/obs/agentview_depth/"][()]
    else:
        depth = np.zeros_like(rgb)
    data_config = json.loads(demo_file["data"].attrs['data_config'])
    K = data_config["intrinsics"]["camera_rs_0"]
    
masks = np.load(os.path.join(annotation_folder, "masks.npz"))['arr_0']
mask_indices = sorted(np.unique(masks))[1:] # dont use 0 because it is the background

all_ob_masks = []

for i in range(len(mask_indices)):
    ob_specific_mask = masks.copy()
    ob_specific_mask[ob_specific_mask != mask_indices[i]] = 0
    ob_specific_mask[ob_specific_mask == mask_indices[i]] = 255
    
    all_ob_masks.append(ob_specific_mask)
    display(Image.fromarray(ob_specific_mask[0]))

In [3]:
foundation_pose_dir = os.path.join(annotation_folder, "FoundationPose")
os.makedirs(foundation_pose_dir, exist_ok=True)

rgb_dir = os.path.join(foundation_pose_dir, "rgb")
if os.path.exists(rgb_dir):
    shutil.rmtree(rgb_dir)
os.makedirs(rgb_dir)

for i in range(rgb.shape[0]):
    Image.fromarray(rgb[i]).save(os.path.join(rgb_dir, f"rgb_{i:04d}.png"))

depth_dir = os.path.join(foundation_pose_dir, "depth")
if os.path.exists(depth_dir):
    shutil.rmtree(depth_dir)
os.makedirs(depth_dir)
for i in range(depth.shape[0]):
    cv2.imwrite(os.path.join(depth_dir, f"depth_{i:04d}.png"), depth[i])

for i, mask_index in enumerate(mask_indices):
    mask_dir = os.path.join(foundation_pose_dir, f"ob_{mask_index}_masks")
    if os.path.exists(mask_dir):
        shutil.rmtree(mask_dir)
    os.makedirs(mask_dir)
    for j in range(all_ob_masks[i].shape[0]):
        cv2.imwrite(os.path.join(mask_dir, f"ob_{mask_index}_masks_{j:04d}.png"), all_ob_masks[i][j])



In [ ]:
K = np.zeros((3, 3))
K[0,0] = data_config["intrinsics"]["camera_rs_0"]["fx"]
K[1,1] = data_config["intrinsics"]["camera_rs_0"]["fy"]
K[0,2] = data_config["intrinsics"]["camera_rs_0"]["cx"]
K[1,2] = data_config["intrinsics"]["camera_rs_0"]["cy"]
K[2,2] = 1

#write k to txt
np.savetxt(os.path.join(foundation_pose_dir, "cam_K.txt"), K, fmt='%.6f')
print(f"{K=}")

In [5]:
if has_depth:
    img_idx = 0
    depth_tst_np = depth[img_idx]
    # get indices where depth_tst_np is nonzero
    nonzero_indices = np.nonzero(depth[img_idx])

    cv2.circle(depth_tst_np, (nonzero_indices[1][0], nonzero_indices[0][0]), 10, (255, 255, 255))
    cv2.circle(depth_tst_np, (300,300), 10, (255, 255, 255))

    #print unique depth values and their frequency for each mask index
    for i, mask_index in enumerate(mask_indices):
        unique_depths, counts = np.unique(depth_tst_np[all_ob_masks[i][img_idx] == 255], return_counts=True)

        #print top and bottom 5 depth values
        print(f"For mask index {mask_index}")
        print("Top 5 depth values: ", unique_depths[-5:])
        print("Bottom 5 depth values: ", unique_depths[:5])
        print(sorted(zip(unique_depths, counts), reverse=True, key=lambda x: x[1]))
        print("---------------------------------------")

    img = Image.fromarray(depth_tst_np * 255)
    display(img)

In [ ]:
depth_tst = Image.open("/home/arisrei/research/RPL/ORION/third_party/FoundationPose/demo_data/mustard0/depth/1581120424100262102.png")
depth_tst_np = np.array(depth_tst)
# get indices where depth_tst_np is nonzero
nonzero_indices = np.nonzero(depth_tst_np)

cv2.circle(depth_tst_np, (nonzero_indices[1][0], nonzero_indices[0][0]), 10, (255, 255, 255))
cv2.circle(depth_tst_np, (300,300), 10, (255, 255, 255))
print((nonzero_indices[1][0], nonzero_indices[0][0]))
print(depth_tst_np[nonzero_indices[0][0],nonzero_indices[1][0]])
print(np.max(depth_tst_np))
img = Image.fromarray(depth_tst_np * 20)
display(img)